# ð Análise Exploratória de Dados (EDA)
## Desafio Synapsee — Telco Customer Churn

**Objetivo:** Compreender o perfil dos clientes que cancelam o serviço (Churn), identificar problemas de qualidade nos dados e mapear as variáveis com maior poder preditivo.

---

## 1. Setup e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Caminhos absolutos baseados na raiz do repositório.
# Resolve qualquer diretório de trabalho (notebook, terminal, CI).
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
DATA_PATH = REPO_ROOT / 'data' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
FIGURES_DIR = REPO_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raiz do reposit\u00f3rio: {REPO_ROOT}')
print(f'Arquivo de dados:    {DATA_PATH}')
print(f'Diret\u00f3rio de figuras: {FIGURES_DIR}')
print()

df_raw = pd.read_csv(DATA_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 2. Visão Geral e Tipos de Dados

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include='all').T

In [ ]:
# Contagem de valores únicos por coluna
print('Valores \u00fanicos por coluna:')
print(df_raw.nunique().to_string())

## 3. Detecção de Nulos Ocultos

**Hipótese mapeada no PLANO:** A coluna `TotalCharges` pode conter espaços em branco (strings vazias) que impedem a conversão para numérico, mascarando valores nulos.

In [ ]:
# 3.1 - Nulos explícitos (NaN) no DataFrame original
print('=== Nulos expl\u00edcitos (isnull) ===')
nulos_explicitos = df_raw.isnull().sum()
print(nulos_explicitos[nulos_explicitos > 0] if nulos_explicitos.sum() > 0 else 'Nenhum NaN expl\u00edcito encontrado.')
print()

In [ ]:
# 3.2 - Investigação: TotalCharges é string quando deveria ser numérica
# pandas 3.x usa dtype 'str' nativo ao invés de 'object'
print(f'Tipo de TotalCharges: {df_raw["TotalCharges"].dtype}')
print()

# Detectar linhas com espaço em branco
mascara_espacos = df_raw['TotalCharges'].str.strip() == ''
n_espacos = mascara_espacos.sum()
print(f'Linhas com espa\u00e7o em branco em TotalCharges: {n_espacos}')
print()

if n_espacos > 0:
    print('Registros afetados:')
    display(df_raw[mascara_espacos][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

In [ ]:
# 3.3 - Diagnóstico: Esses clientes têm tenure == 0 (recém-chegados)?
if n_espacos > 0:
    print('Distribui\u00e7\u00e3o de tenure nos registros com TotalCharges vazio:')
    print(df_raw.loc[mascara_espacos, 'tenure'].value_counts())
    print()
    print('>>> Conclus\u00e3o: Clientes com tenure=0 n\u00e3o tiveram cobran\u00e7a acumulada,\n'
          '    logo TotalCharges vazio \u00e9 esperado (nulo sem\u00e2ntico, n\u00e3o erro de cadastro).')

## 4. Análise de Desbalanceamento da Target (`Churn`)

In [ ]:
# 4.1 - Distribuição absoluta e percentual
churn_counts = df_raw['Churn'].value_counts()
churn_pct = df_raw['Churn'].value_counts(normalize=True) * 100

print('Distribui\u00e7\u00e3o da Target (Churn):')
print(pd.DataFrame({'Contagem': churn_counts, 'Percentual (%)': churn_pct.round(2)}))
print(f'\nRaz\u00e3o de desbalanceamento (No/Yes): {churn_counts["No"] / churn_counts["Yes"]:.2f}:1')

In [ ]:
# 4.2 - Visualização do desbalanceamento
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#2ecc71', '#e74c3c']

# Gráfico de barras
sns.countplot(data=df_raw, x='Churn', ax=axes[0], palette=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribui\u00e7\u00e3o Absoluta de Churn', fontweight='bold')
axes[0].set_ylabel('Contagem')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold', fontsize=12)

# Gráfico de pizza
axes[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0, 0.05),
            shadow=True, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Propor\u00e7\u00e3o de Churn', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_churn_distribuicao.png', bbox_inches='tight', dpi=150)
plt.show()
print('>>> Dataset desbalanceado: ~73% No vs ~27% Yes.')

## 5. Análise de Features Numéricas vs Churn

In [ ]:
# Preparar TotalCharges como numérica para as análises
# Espaços em branco viram NaN temporariamente (tratamento definitivo na seção 10)
df_eda = df_raw.copy()
df_eda['TotalCharges'] = pd.to_numeric(df_eda['TotalCharges'], errors='coerce')

numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numericas):
    sns.boxplot(data=df_eda, x='Churn', y=col, ax=axes[i], palette=colors,
                linewidth=1.2, fliersize=3)
    axes[i].set_title(f'{col} vs Churn', fontweight='bold')

plt.suptitle('Distribui\u00e7\u00e3o de Vari\u00e1veis Num\u00e9ricas por Churn', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_numericas_vs_churn.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 5.2 - Distribuições (KDE) lado a lado por Churn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numericas):
    for label, color in zip(['No', 'Yes'], colors):
        subset = df_eda[df_eda['Churn'] == label][col].dropna()
        sns.kdeplot(subset, ax=axes[i], label=f'Churn={label}', color=color, linewidth=2, fill=True, alpha=0.3)
    axes[i].set_title(f'Distribui\u00e7\u00e3o de {col}', fontweight='bold')
    axes[i].legend()

plt.suptitle('Densidade (KDE) das Vari\u00e1veis Num\u00e9ricas por Churn', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_kde_numericas.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 5.3 - Estatísticas descritivas numéricas agrupadas por Churn
print('Estat\u00edsticas por classe de Churn:')
display(df_eda.groupby('Churn')[numericas].describe().T)

## 6. Análise de Features Categóricas vs Churn

Vamos calcular a **taxa de churn** por categoria e identificar quais atributos apresentam maior diferença de comportamento entre clientes que cancelam e os que ficam.

In [ ]:
# Identificar colunas categóricas relevantes (excluindo customerID e a target)
# pandas 3.x usa dtype 'str' nativo — usamos is_string_dtype para compatibilidade
categoricas = [col for col in df_eda.columns
               if pd.api.types.is_string_dtype(df_eda[col])
               and col not in ['customerID', 'Churn']]

print(f'Features categ\u00f3ricas identificadas ({len(categoricas)}):')
print(categoricas)

In [ ]:
# 6.1 - Taxa de churn por categoria (visualização em grid)
n_cols_grid = 4
n_rows_grid = max(1, (len(categoricas) + n_cols_grid - 1) // n_cols_grid)

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(20, n_rows_grid * 4.5))
axes = axes.flatten()

for i, col in enumerate(categoricas):
    churn_rate = df_eda.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    churn_rate = churn_rate.sort_values(ascending=False)

    bars = axes[i].barh(churn_rate.index, churn_rate.values, color='#e74c3c', edgecolor='black', linewidth=0.5, alpha=0.85)
    axes[i].set_title(f'{col}', fontweight='bold', fontsize=11)
    axes[i].set_xlabel('Taxa de Churn (%)')
    axes[i].set_xlim(0, 60)
    axes[i].axvline(x=churn_pct['Yes'], color='gray', linestyle='--', alpha=0.7,
                    label=f'M\u00e9dia global ({churn_pct["Yes"]:.1f}%)')

    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
                     va='center', fontsize=9, fontweight='bold')

for j in range(len(categoricas), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Taxa de Churn por Feature Categ\u00f3rica (linha tracejada = m\u00e9dia global)',
             fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_categoricas_vs_churn.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Ranking de Relevância: Features com Maior Peso no Churn

Ranking consolidado:
- **Numéricas:** Correlação ponto-bisserial com Churn (binary encoded)
- **Categóricas:** Cramér's V para medir a associação com Churn

In [ ]:
from scipy.stats import pointbiserialr, chi2_contingency

churn_binary = (df_eda['Churn'] == 'Yes').astype(int)

print('=== Correla\u00e7\u00e3o Ponto-Bisserial (Num\u00e9ricas vs Churn) ===')
resultados_num = []
for col in numericas:
    valid = df_eda[col].dropna()
    corr, pval = pointbiserialr(churn_binary[valid.index], valid)
    resultados_num.append({'Feature': col, 'Correla\u00e7\u00e3o': round(corr, 4), 'p-valor': f'{pval:.2e}'})

df_corr_num = pd.DataFrame(resultados_num).sort_values('Correla\u00e7\u00e3o', key=abs, ascending=False)
display(df_corr_num)
print()

In [ ]:
def cramers_v(x, y):
    """Calcula Cramer's V para medir associacao entre duas variaveis categoricas."""
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    min_dim = min(confusion_matrix.shape) - 1
    return np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0

print("=== Cramer's V (Categoricas vs Churn) ===")
resultados_cat = []
for col in categoricas:
    v = cramers_v(df_eda[col], df_eda['Churn'])
    resultados_cat.append({'Feature': col, "Cramer's V": round(v, 4)})

df_cramer = pd.DataFrame(resultados_cat).sort_values("Cramer's V", ascending=False)
display(df_cramer)

In [ ]:
# 7.1 - Ranking visual consolidado
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_corr_plot = df_corr_num.sort_values('Correla\u00e7\u00e3o', key=abs, ascending=True)
bar_colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in df_corr_plot['Correla\u00e7\u00e3o']]
axes[0].barh(df_corr_plot['Feature'], df_corr_plot['Correla\u00e7\u00e3o'], color=bar_colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Correla\u00e7\u00e3o Ponto-Bisserial\n(Num\u00e9ricas vs Churn)', fontweight='bold')
axes[0].set_xlabel('Correla\u00e7\u00e3o')
axes[0].axvline(x=0, color='gray', linewidth=0.8)
for idx, row in df_corr_plot.iterrows():
    axes[0].text(row['Correla\u00e7\u00e3o'], row['Feature'], f"  {row['Correla\u00e7\u00e3o']:.3f}", va='center', fontweight='bold', fontsize=10)

df_cramer_plot = df_cramer.sort_values("Cramer's V", ascending=True)
axes[1].barh(df_cramer_plot['Feature'], df_cramer_plot["Cramer's V"], color='#3498db', edgecolor='black', linewidth=0.8)
axes[1].set_title("Cramer's V\n(Categoricas vs Churn)", fontweight='bold')
axes[1].set_xlabel("Cramer's V")
for idx, row in df_cramer_plot.iterrows():
    axes[1].text(row["Cramer's V"], row['Feature'], f"  {row[chr(67)+'ramer'+chr(39)+'s V']:.3f}", va='center', fontweight='bold', fontsize=10)

plt.suptitle('Ranking de Relev\u00e2ncia das Features para Previs\u00e3o de Churn', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_ranking_features.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Correlação entre Variáveis Numéricas (Heatmap)

In [ ]:
df_corr_matrix = df_eda[numericas + ['SeniorCitizen']].copy()
df_corr_matrix['Churn_binary'] = churn_binary

fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df_corr_matrix.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correla\u00e7\u00e3o (Pearson)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_heatmap_correlacao.png', bbox_inches='tight', dpi=150)
plt.show()

## 9. Varredura Sistêmica de Qualidade de Dados

Checagem abrangente em **todas** as colunas do dataset, procurando por:
- Espaços em branco, strings vazias, marcadores de nulo disfarcçados (`?`, `NA`, `N/A`, `null`, `-`)
- Outliers ilógicos nas variáveis numéricas (`tenure` negativo, cobranças negativas, etc.)

In [ ]:
MARCADORES_NULO = ['', ' ', '?', 'NA', 'N/A', 'null', 'NULL', 'None', 'none', '-', '--', 'nan', 'NaN']

colunas_texto = [col for col in df_raw.columns if pd.api.types.is_string_dtype(df_raw[col])]

print('=== Varredura de Anomalias em Colunas de Texto ===')
print(f'Colunas verificadas: {len(colunas_texto)}')
print()

anomalias_encontradas = {}
for col in colunas_texto:
    resultados_coluna = {}
    for marcador in MARCADORES_NULO:
        contagem = (df_raw[col].str.strip() == marcador).sum()
        if contagem > 0:
            label = repr(marcador) if marcador.strip() == '' else marcador
            resultados_coluna[label] = contagem
    if resultados_coluna:
        anomalias_encontradas[col] = resultados_coluna

if anomalias_encontradas:
    print('Anomalias detectadas:')
    for col, detalhes in anomalias_encontradas.items():
        print(f'  {col}:')
        for marcador, contagem in detalhes.items():
            print(f'      {marcador}: {contagem} ocorrencias')
else:
    print('Nenhuma anomalia de texto encontrada alem das ja mapeadas (TotalCharges).')

print()

In [ ]:
print('=== Validacao de Integridade Numerica ===')
print()

variaveis_numericas_completas = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
stats = df_eda[variaveis_numericas_completas].describe()
display(stats)
print()

print('--- Checagem de Valores Negativos ---')
for col in variaveis_numericas_completas:
    negativos = (df_eda[col].dropna() < 0).sum()
    status = 'ALERTA' if negativos > 0 else 'OK'
    print(f'  {col}: {negativos} valores negativos [{status}]')
print()

print('--- Checagem de Dominio Logico ---')
senior_invalidos = (~df_eda['SeniorCitizen'].isin([0, 1])).sum()
print(f'  SeniorCitizen fora de (0,1): {senior_invalidos} [{"ALERTA" if senior_invalidos > 0 else "OK"}]')

tenure_max = df_eda['tenure'].max()
print(f'  tenure maximo: {tenure_max} meses ({tenure_max/12:.1f} anos) [{"ALERTA" if tenure_max > 120 else "OK"}]')

monthly_zero = (df_eda['MonthlyCharges'] == 0).sum()
print(f'  MonthlyCharges == 0: {monthly_zero} [{"ALERTA" if monthly_zero > 0 else "OK"}]')

duplicatas_id = df_raw['customerID'].duplicated().sum()
print(f'  Duplicatas de customerID: {duplicatas_id} [{"ALERTA" if duplicatas_id > 0 else "OK"}]')

print()
print('>>> Conclusao: Dataset sem anomalias numericas. Valores dentro dos limites logicos esperados.')

---
## 10. Transformacoes Aprovadas (Ponto de Controle Validado)

Decisoes aprovadas pelo usuario apos a EDA:
1. **Imputar TotalCharges** com `0` nos 11 registros com `tenure=0`
2. **Criar `TicketMedio`** = `TotalCharges / tenure` (para `tenure > 0`; `0` quando `tenure == 0`)
3. **Dropar `TotalCharges`** original para mitigar multicolinearidade com `tenure`
4. **Balanceamento por `class_weight='balanced'`** sera adotado na etapa de modelagem

In [ ]:
# 10.1 - Imputar TotalCharges com 0 nos registros com espaco em branco (tenure=0)
# Justificativa: Sao clientes recem-chegados sem cobranca acumulada (nulo semantico)
df_eda['TotalCharges'] = pd.to_numeric(df_eda['TotalCharges'], errors='coerce').fillna(0)

print(f'NaN restantes em TotalCharges: {df_eda["TotalCharges"].isnull().sum()}')
print(f'Registros com TotalCharges == 0: {(df_eda["TotalCharges"] == 0).sum()}')
print(f'Tenure desses registros: {df_eda.loc[df_eda["TotalCharges"] == 0, "tenure"].unique()}')

In [ ]:
# 10.2 - Criar feature TicketMedio: gasto medio mensal real do cliente
# Formula: TotalCharges / tenure (quando tenure > 0), senao 0
# Objetivo: Substituir TotalCharges por metrica sem multicolinearidade com tenure
df_eda['TicketMedio'] = np.where(
    df_eda['tenure'] > 0,
    df_eda['TotalCharges'] / df_eda['tenure'],
    0
)

print('Estatisticas de TicketMedio:')
print(df_eda['TicketMedio'].describe())
print()
print('TicketMedio medio por classe de Churn:')
print(df_eda.groupby('Churn')['TicketMedio'].mean())

In [ ]:
# 10.3 - Dropar TotalCharges original (mitigar multicolinearidade com tenure)
df_eda = df_eda.drop(columns=['TotalCharges'])

print(f'Shape final de df_eda: {df_eda.shape}')
print(f'Colunas: {list(df_eda.columns)}')

## 11. Resumo Executivo da EDA

### Achados Principais:

1. **Nulos Ocultos:** `TotalCharges` continha 11 espacos em branco (clientes com `tenure=0`). Imputados com `0` (decisao aprovada).

2. **Qualidade Geral:** Varredura sistemica confirmou zero anomalias adicionais. Sem `?`, `NA`, strings corrompidas. Sem valores negativos, sem duplicatas de IDs.

3. **Desbalanceamento:** ~26.5% Churn=Yes (ratio 2.77:1). Estrategia: `class_weight='balanced'` na modelagem.

4. **Features Numericas com Forte Sinal:**
   - `tenure` (correlacao -0.35): Clientes antigos cancelam muito menos
   - `MonthlyCharges` (+0.19): Cobrancas altas associadas a maior churn

5. **Features Categoricas com Maior Peso (Cramer's V):**
   - `Contract` (Month-to-month tem churn altissimo)
   - `OnlineSecurity` / `TechSupport` (ausencia eleva churn)
   - `InternetService` (Fiber optic tem taxa elevada)
   - `PaymentMethod` (Electronic check concentra mais churn)

6. **Feature Engineering:** `TicketMedio` criada (TotalCharges/tenure). `TotalCharges` original dropada (multicolinearidade com tenure ~0.83).

### Proximos Passos (Etapa 2 - Pipeline de Features):
- Encoding de variaveis categoricas
- Escalonamento de numericas
- Selecao de features